# Notebook 03: Multi-task MLP with Full Optimizations

- Multi-task shared backbone (128→64) + 3 classification heads + 1 RGB regression head
- Focal loss (γ=2) + class weights + AdamW + ReduceLROnPlateau + early stopping
- Ingredient TF-IDF (min_df=20, max_features=300)
- Domain ratio features (SiO2/Al2O3, fluxes, Si/Alkali, etc.)
- 5-fold CV + softmax ensemble on test set
- No hardcoded oxide lists — everything auto-derived from data


In [8]:
import sys, subprocess, warnings, os, re, json, time
import numpy as np
import pandas as pd

try:
    import torch
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch', '--index-url', 'https://download.pytorch.org/whl/cu121'])
    import torch

try:
    import xgboost
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', 'scikit-learn'])

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  Device: {DEVICE}')

PyTorch 2.9.1+gitff65f5b  Device: cuda


## 1. Load raw data


In [9]:
DATA_DIR = 'data/property_prediction'
assert os.path.exists(DATA_DIR), f'Upload data to {DATA_DIR}'

def jload(path):
    with open(os.path.join(DATA_DIR, path)) as f:
        return json.load(f)

train_recipes = jload('train/recipes.json')
train_targets = jload('train/targets.json')
test_recipes  = jload('test/recipes.json')
test_targets  = jload('test/targets.json')
print(f'Train: {len(train_recipes)} recipes / {len(train_targets)} targets')
print(f'Test:  {len(test_recipes)} recipes / {len(test_targets)} targets')

Train: 16781 recipes / 16781 targets
Test:  4903 recipes / 4903 targets


## 2. Build flat DataFrame (all features auto-derived)


In [10]:
def build_df(recipes, targets):
    tmap = {t['id']: t for t in targets}
    rows = []
    for r in recipes:
        t = tmap.get(r['id'])
        if t is None:
            continue
        row = {'id': r['id']}
        # UMF oxides
        umf = r.get('umf') or {}
        for k, v in umf.items():
            row['umf_' + k] = v
        # Chemical composition
        cc = r.get('chemical_composition') or {}
        for k, v in cc.items():
            if isinstance(v, (int, float)):
                row['chem_' + k] = v
        # Cone
        row['cone_min'] = r.get('cone_min')
        row['cone_max'] = r.get('cone_max')
        # Atmosphere
        row['atmosphere'] = r.get('atmosphere', '')
        # Misc
        row['has_chem'] = float(r.get('has_chemical_data', False))
        row['ing_count'] = r.get('ingredient_count', 0)
        row['batch_total'] = cc.get('Amt.', 0) or 0
        row['loi'] = cc.get('LOI', 0) or 0
        # Ingredients
        ings = r.get('ingredients', [])
        row['ing_text'] = ' '.join(ing.get('material', '') for ing in ings)
        # Targets
        for h in ('surface', 'transparency', 'color_family'):
            v = t.get(h)
            if v is not None:
                row[h] = v
        # Color RGB
        rgb = t.get('color_rgb')
        if rgb:
            row['color_r'] = rgb.get('r') / 255.0
            row['color_g'] = rgb.get('g') / 255.0
            row['color_b'] = rgb.get('b') / 255.0
        rows.append(row)
    return pd.DataFrame(rows)

df_train = build_df(train_recipes, train_targets)
df_test  = build_df(test_recipes,  test_targets)

# Drop rows with no UMF data
umf_cols = [c for c in df_train.columns if c.startswith('umf_')]
df_train = df_train[df_train[umf_cols].notna().any(axis=1)].reset_index(drop=True)
df_test  = df_test[df_test[[c for c in df_test.columns if c.startswith('umf_')]].notna().any(axis=1)].reset_index(drop=True)
print(f'Train: {df_train.shape}  Test: {df_test.shape}')

Train: (15668, 79)  Test: (4849, 69)


### 2a. Ingredient TF-IDF


In [11]:
all_ing = pd.concat([df_train['ing_text'], df_test['ing_text']])
vec = CountVectorizer(token_pattern=r"(?u)\b\w+['\-]?\w+\b", min_df=20, max_features=300)
vec.fit(all_ing)
for df in (df_train, df_test):
    mat = vec.transform(df['ing_text']).toarray()
    for i, c in enumerate(vec.get_feature_names_out()):
        df['ing_' + c] = mat[:, i]
print(f'Ingredient features: {len(vec.get_feature_names_out())}')

Ingredient features: 300


### 2b. Domain chemistry ratios


In [12]:
def add_domain_ratios(df):
    def g(k):
        c = 'umf_' + k
        return df[c] if c in df.columns else 0.0
    si, al = g('SiO2').clip(lower=0), g('Al2O3').clip(lower=1e-6)
    df['r_si_al'] = si / al
    flux_oxides = ['Na2O','K2O','CaO','MgO','ZnO','Li2O','PbO','BaO','SrO']
    fcols = [c for c in df.columns if c.startswith('umf_') and c[4:] in flux_oxides]
    df['r_total_flux'] = df[fcols].sum(axis=1) if fcols else 0.0
    alkali  = ['Na2O','K2O','Li2O']
    ae      = ['CaO','MgO','BaO','SrO']
    acols  = [c for c in df.columns if c.startswith('umf_') and c[4:] in alkali]
    aecols = [c for c in df.columns if c.startswith('umf_') and c[4:] in ae]
    df['r_alkali']       = df[acols].sum(axis=1) if acols else 0.0
    df['r_alkali_earth'] = df[aecols].sum(axis=1) if aecols else 0.0
    at = df['r_alkali'] + df['r_alkali_earth'].clip(lower=1e-6)
    df['r_alkali_ratio'] = df['r_alkali'] / at
    na = g('Na2O').clip(lower=1e-6)
    df['r_na_k'] = g('K2O') / na
    colorants = ['Fe2O3','CuO','CoO','Cr2O3','MnO2','TiO2','MnO','NiO','V2O5']
    ccols = [c for c in df.columns if c.startswith('umf_') and c[4:] in colorants]
    df['r_colorant_load'] = df[ccols].sum(axis=1) if ccols else 0.0
    tf = df['r_total_flux'].clip(lower=1e-6)
    df['r_si_flux'] = si / tf
    df['r_b2o3_present'] = (g('B2O3') > 0).astype(float)
    df['r_pbo_present']  = (g('PbO') > 0).astype(float)
    df['r_fe_total']     = g('Fe2O3')
    df.fillna(0.0, inplace=True)

add_domain_ratios(df_train)
add_domain_ratios(df_test)
r_cols = [c for c in df_train.columns if c.startswith('r_')]
print(f'Domain ratios: {len(r_cols)}')

Domain ratios: 11


### 2c. Atmosphere encoding + cone clipping


In [13]:
for df in (df_train, df_test):
    a = df['atmosphere'].str.lower()
    df['atmos_oxidation'] = a.str.contains('oxidation', na=False).astype(float)
    df['atmos_reduction'] = a.str.contains('reduction', na=False).astype(float)
    df['atmos_neutral']   = a.str.contains('neutral', na=False).astype(float)
    df['cone_max'] = df['cone_max'].clip(upper=22)
    df['cone_min'] = df['cone_min'].clip(upper=22)

### 2d. Assemble feature column list


In [14]:
FEATURE_COLS = (
    [c for c in df_train.columns if c.startswith('umf_')] +
    [c for c in df_train.columns if c.startswith('ing_') and c != 'ing_text'] +
    [c for c in df_train.columns if c.startswith('r_')] +
    ['atmos_oxidation','atmos_reduction','atmos_neutral',
     'cone_min','cone_max','has_chem','ing_count','batch_total','loi']
)
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_train.columns and df_train[c].dtype in (np.float64, np.float32, np.int64, np.int32)]
print(f'Total features: {len(FEATURE_COLS)}')
print(f'  UMF: {sum(1 for c in FEATURE_COLS if c.startswith("umf_"))}')
print(f'  Ing: {sum(1 for c in FEATURE_COLS if c.startswith("ing_"))}')
print(f'  Dom: {sum(1 for c in FEATURE_COLS if c.startswith("r_"))}')
print(f'  Other: {sum(1 for c in FEATURE_COLS if not c.startswith(("umf_","ing_","r_")))}')

Total features: 339
  UMF: 18
  Ing: 302
  Dom: 11
  Other: 8


### 2e. Fill missing & scale


In [15]:
scaler = StandardScaler()
for col in FEATURE_COLS:
    if col not in df_train.columns: df_train[col] = 0.0
    if col not in df_test.columns:  df_test[col]  = 0.0
    df_train[col] = df_train[col].fillna(0.0)
    df_test[col]  = df_test[col].fillna(0.0)
df_train[FEATURE_COLS] = scaler.fit_transform(df_train[FEATURE_COLS])
df_test[FEATURE_COLS]  = scaler.transform(df_test[FEATURE_COLS])
print('Features scaled')

Features scaled


### 2f. Encode targets + class weights


In [16]:
HEADS = ['surface', 'transparency', 'color_family']
hd = {}  # head_datasets
hw = {}  # head_weights

for h in HEADS:
    mask_tr = df_train[h].notna() & df_train[h].apply(lambda x: isinstance(x, str))
    mask_te = df_test[h].notna() & df_test[h].apply(lambda x: isinstance(x, str))
    le = LabelEncoder()
    y_tr = le.fit_transform(df_train.loc[mask_tr, h])
    y_te = le.transform(df_test.loc[mask_te, h])
    counts = np.bincount(y_tr)
    w = len(y_tr) / (len(counts) * counts.astype(float))
    w = w / w.sum() * len(counts)
    hd[h] = {
        'le': le,
        'y_tr': y_tr, 'y_te': y_te,
        'n': len(le.classes_),
        'classes': le.classes_,
        'tr_idx': np.where(mask_tr.values)[0],
        'te_idx': np.where(mask_te.values)[0],
    }
    hw[h] = torch.tensor(w, dtype=torch.float)
    print(f'{h:15s}  train={len(y_tr):5d}  test={len(y_te):4d}  classes={list(le.classes_)}')

# RGB auxiliary (regression)
mask = df_train[['color_r','color_g','color_b']].notna().all(axis=1)
mask_t = df_test[['color_r','color_g','color_b']].notna().all(axis=1)
hd['color_rgb'] = {
    'X_tr': df_train.loc[mask, FEATURE_COLS].values.astype(np.float32),
    'y_tr': np.stack([df_train.loc[mask, c].values for c in ('color_r','color_g','color_b')], axis=1).astype(np.float32),
    'X_te': df_test.loc[mask_t, FEATURE_COLS].values.astype(np.float32),
    'y_te': np.stack([df_test.loc[mask_t, c].values for c in ('color_r','color_g','color_b')], axis=1).astype(np.float32),
}
print(f'color_rgb      train={mask.sum():5d}  test={mask_t.sum():4d}')

surface          train= 9367  test=3728  classes=['Dry Matte', 'Glossy', 'Matte', 'Satin', 'Satin-matte', 'Semi-glossy', 'Semi-matte', 'Smooth Matte', 'Stony Matte']
transparency     train= 9013  test=3320  classes=['Opaque', 'Semi-opaque', 'Translucent', 'Transparent']
color_family     train=15668  test=4849  classes=['Black', 'Blue', 'Gray', 'Green', 'Orange', 'Purple', 'Red', 'White', 'Yellow']
color_rgb      train=15668  test=4849


## 3. Model definition


In [17]:
class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
    def forward(self, input, target):
        logpt = -torch.nn.functional.nll_loss(input, target, weight=self.weight, reduction='none')
        pt = torch.exp(logpt)
        return ((1 - pt) ** self.gamma * -logpt).mean()

class MultiTaskMLP(torch.nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.shared = torch.nn.Sequential(
            torch.nn.Linear(in_dim, 128),
            torch.nn.BatchNorm1d(128), torch.nn.ReLU(), torch.nn.Dropout(0.3),
            torch.nn.Linear(128, 64),
            torch.nn.BatchNorm1d(64),  torch.nn.ReLU(), torch.nn.Dropout(0.2),
        )
        self.surface_head      = torch.nn.Linear(64, hd['surface']['n'])
        self.transparency_head = torch.nn.Linear(64, hd['transparency']['n'])
        self.color_head        = torch.nn.Linear(64, hd['color_family']['n'])
        self.rgb_head          = torch.nn.Linear(64, 3)
    def forward(self, x):
        h = self.shared(x)
        s = torch.nn.functional.log_softmax(self.surface_head(h), dim=-1)
        t = torch.nn.functional.log_softmax(self.transparency_head(h), dim=-1)
        c = torch.nn.functional.log_softmax(self.color_head(h), dim=-1)
        r = torch.sigmoid(self.rgb_head(h))
        return s, t, c, r

## 4. Training with 5-fold CV


In [18]:
N_FOLDS = 5
N_EPOCHS = 200
BATCH_SIZE = 128
PATIENCE = 15
LR = 1e-3

# Full training array for fold splitting
X_full = df_train[FEATURE_COLS].values.astype(np.float32)
y_full_strat = df_train['color_family'].values  # stratify by color_family

# Build full label arrays with -100 masking
def make_label_array(head):
    arr = np.full(len(df_train), -100, dtype=np.int64)
    arr[hd[head]['tr_idx']] = hd[head]['y_tr']
    return arr

label_arrays = {h: make_label_array(h) for h in HEADS}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
cv_models = []
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_full_strat)):
    print(f'\n{"="*60}\n  Fold {fold+1}/{N_FOLDS}\n{"="*60}')

    model = MultiTaskMLP(len(FEATURE_COLS)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

    # Build train/val tensors for each head (-100 for missing in this fold)
    def fold_tensors(indices, head):
        x = torch.tensor(X_full[indices])
        y = torch.tensor(label_arrays[head][indices])
        return x, y

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_full[tr_idx]),
            *[torch.tensor(label_arrays[h][tr_idx]) for h in HEADS]
        ),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    val_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_full[va_idx]),
            *[torch.tensor(label_arrays[h][va_idx]) for h in HEADS]
        ),
        batch_size=BATCH_SIZE, shuffle=False)

    best_f1 = -1
    best_state = None
    patience = 0

    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            x = batch[0].to(DEVICE)
            s, t, c, r = model(x)
            outs = [s, t, c]
            loss = 0
            n_valid = 0
            for i, h in enumerate(HEADS):
                y = batch[1 + i].to(DEVICE)
                valid = y != -100
                if valid.sum() == 0:
                    continue
                loss += FocalLoss(gamma=2.0, weight=hw[h].to(DEVICE))(outs[i][valid], y[valid])
                n_valid += 1
            if n_valid == 0:
                continue
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        val_preds = {h: [] for h in HEADS}
        val_labels = {h: [] for h in HEADS}
        with torch.no_grad():
            for batch in val_loader:
                x = batch[0].to(DEVICE)
                s, t, c, r = model(x)
                outs = [s, t, c]
                for i, h in enumerate(HEADS):
                    y = batch[1 + i]
                    valid = y != -100
                    if valid.sum() == 0:
                        continue
                    val_preds[h].append(outs[i][valid].argmax(dim=-1).cpu().numpy())
                    val_labels[h].append(y[valid].numpy())

        val_metrics = {}
        for h in HEADS:
            if len(val_preds[h]) == 0:
                continue
            yp = np.concatenate(val_preds[h])
            yt = np.concatenate(val_labels[h])
            val_metrics[h] = {
                'accuracy': accuracy_score(yt, yp),
                'macro_f1': f1_score(yt, yp, average='macro'),
            }

        avg_f1 = np.mean([m['macro_f1'] for m in val_metrics.values()])

        if avg_f1 > best_f1:
            best_f1 = avg_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1

        scheduler.step(avg_f1)

        if epoch == 1 or epoch % 10 == 0:
            detail = '  '.join(f'{h}: acc={m["accuracy"]:.4f} f1={m["macro_f1"]:.4f}' for h, m in val_metrics.items())
            print(f'  epoch {epoch:3d}  loss={total_loss:.4f}  {detail}  best_f1={best_f1:.4f}')

        if patience >= PATIENCE:
            print(f'  early stop at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    cv_models.append(model.cpu())
    fold_scores.append({'fold': fold, **val_metrics})
    print(f'  Fold {fold+1} done: avg_f1={best_f1:.4f}')


  Fold 1/5
  epoch   1  loss=124.0329  surface: acc=0.0995 f1=0.1077  transparency: acc=0.2865 f1=0.2898  color_family: acc=0.0670 f1=0.0651  best_f1=0.1542
  epoch  10  loss=77.3917  surface: acc=0.1267 f1=0.1499  transparency: acc=0.3475 f1=0.3572  color_family: acc=0.1047 f1=0.0950  best_f1=0.2013
  epoch  20  loss=64.2207  surface: acc=0.1385 f1=0.1633  transparency: acc=0.3632 f1=0.3726  color_family: acc=0.1181 f1=0.1052  best_f1=0.2211
  epoch  30  loss=57.7477  surface: acc=0.1497 f1=0.1769  transparency: acc=0.4012 f1=0.4091  color_family: acc=0.1315 f1=0.1252  best_f1=0.2371
  epoch  40  loss=52.9077  surface: acc=0.1428 f1=0.1699  transparency: acc=0.4085 f1=0.4155  color_family: acc=0.1308 f1=0.1235  best_f1=0.2410
  epoch  50  loss=46.2205  surface: acc=0.1519 f1=0.1825  transparency: acc=0.4169 f1=0.4277  color_family: acc=0.1398 f1=0.1299  best_f1=0.2467
  epoch  60  loss=44.2436  surface: acc=0.1455 f1=0.1775  transparency: acc=0.4236 f1=0.4348  color_family: acc=0.134

## 5. Cross-validation summary


In [19]:
print(f'{"Head":<15s} {"Accuracy":>10s} {"Macro F1":>10s}')
print('-' * 40)
for h in HEADS:
    accs = [f[h]['accuracy'] for f in fold_scores if h in f]
    f1s  = [f[h]['macro_f1'] for f in fold_scores if h in f]
    print(f'{h:<15s}  {np.mean(accs):.4f}\u00b1{np.std(accs):.4f}  {np.mean(f1s):.4f}\u00b1{np.std(f1s):.4f}')
all_f1 = [f[h]['macro_f1'] for f in fold_scores for h in HEADS if h in f]
print(f'{"Average":<15s}  {np.mean(all_f1):.4f}')

Head              Accuracy   Macro F1
----------------------------------------
surface          0.1518±0.0021  0.1748±0.0076
transparency     0.4268±0.0073  0.4378±0.0070
color_family     0.1372±0.0053  0.1259±0.0056
Average          0.2461


## 6. Test evaluation (5-fold ensemble)


In [20]:
X_test_t = torch.tensor(df_test[FEATURE_COLS].values.astype(np.float32))

test_results = {}
for h in HEADS:
    probs = []
    for m in cv_models:
        m = m.to(DEVICE)
        m.eval()
        with torch.no_grad():
            s, t, c, r = m(X_test_t.to(DEVICE))
            out_map = {'surface': s, 'transparency': t, 'color_family': c}
            probs.append(torch.exp(out_map[h]).cpu().numpy())
        m = m.cpu()
    avg_probs = np.mean(probs, axis=0)
    y_pred = np.argmax(avg_probs, axis=1)
    te_idx = hd[h]['te_idx']
    y_true = hd[h]['y_te']
    test_results[h] = {
        'accuracy': accuracy_score(y_true, y_pred[te_idx]),
        'macro_f1': f1_score(y_true, y_pred[te_idx], average='macro'),
        'weighted_f1': f1_score(y_true, y_pred[te_idx], average='weighted'),
    }

print(f'{"Head":<15s} {"Accuracy":>10s} {"Macro F1":>10s} {"Wtd F1":>10s}')
print('-' * 50)
for h in HEADS:
    r = test_results[h]
    print(f'{h:<15s}  {r["accuracy"]:.4f}  {r["macro_f1"]:.4f}  {r["weighted_f1"]:.4f}')
avg = np.mean([test_results[h]['macro_f1'] for h in HEADS])
print(f'{"Average":<15s}  {np.mean([test_results[h]["accuracy"] for h in HEADS]):.4f}  {avg:.4f}')

Head              Accuracy   Macro F1     Wtd F1
--------------------------------------------------
surface          0.1320  0.1432  0.0988
transparency     0.3702  0.3770  0.3644
color_family     0.1745  0.1475  0.1130
Average          0.2255  0.2226


## 7. Per-class breakdown


In [21]:
for h in HEADS:
    print(f'\n=== {h} ===')
    te_idx = hd[h]['te_idx']
    probs = []
    for m in cv_models:
        m = m.to(DEVICE)
        m.eval()
        with torch.no_grad():
            s, t, c, r = m(X_test_t.to(DEVICE))
            out_map = {'surface': s, 'transparency': t, 'color_family': c}
            probs.append(torch.exp(out_map[h]).cpu().numpy())
        m = m.cpu()
    avg_probs = np.mean(probs, axis=0)
    y_pred = np.argmax(avg_probs, axis=1)[te_idx]
    y_true = hd[h]['y_te']
    print(classification_report(y_true, y_pred, target_names=hd[h]['classes'], digits=4))


=== surface ===
              precision    recall  f1-score   support

   Dry Matte     0.0733    0.3333    0.1202        33
      Glossy     0.0000    0.0000    0.0000      1692
       Matte     0.3063    0.2126    0.2510       461
       Satin     0.1218    0.1610    0.1387       323
 Satin-matte     0.0840    0.2077    0.1197       284
 Semi-glossy     0.1664    0.2338    0.1944       539
  Semi-matte     0.1128    0.3901    0.1751       223
Smooth Matte     0.1004    0.3852    0.1593       122
 Stony Matte     0.0902    0.2353    0.1304        51

    accuracy                         0.1320      3728
   macro avg     0.1173    0.2399    0.1432      3728
weighted avg     0.0908    0.1320    0.0988      3728


=== transparency ===
              precision    recall  f1-score   support

      Opaque     0.7911    0.2107    0.3328      1438
 Semi-opaque     0.2973    0.5283    0.3805       759
 Translucent     0.2725    0.4639    0.3433       595
 Transparent     0.4330    0.4716    0.

## 8. Comparison vs XGBoost baseline


In [26]:
DELTA = chr(916)
print(f'{"Head":<15s} {"MLP Acc":>8s} {"MLP F1":>8s} {"XGB Acc":>8s} {"XGB F1":>8s} {DELTA + " Acc":>8s} {DELTA + " F1":>8s}')
print('-' * 70)

xgb_params = {
    'objective': 'multi:softprob',
    'learning_rate': 0.1, 'max_depth': 6,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'seed': 42,
}
xgb_results = {}

for h in HEADS:
    d = hd[h]
    from xgboost import XGBClassifier
    m = XGBClassifier(
        **xgb_params, n_estimators=150, num_class=d['n'],
        early_stopping_rounds=20, eval_metric='mlogloss',
        random_state=42
    )
    m.fit(
        df_train.loc[d['tr_idx'], FEATURE_COLS].values, d['y_tr'],
        eval_set=[(df_test.loc[d['te_idx'], FEATURE_COLS].values, d['y_te'])],
        verbose=False
    )
    y_pred = m.predict(df_test.loc[d['te_idx'], FEATURE_COLS].values)
    xgb_results[h] = {
        'accuracy': accuracy_score(d['y_te'], y_pred),
        'macro_f1': f1_score(d['y_te'], y_pred, average='macro'),
    }
    r = test_results[h]
    da = r['accuracy'] - xgb_results[h]['accuracy']
    df1 = r['macro_f1'] - xgb_results[h]['macro_f1']
    print(f'{h:<15s}  {r["accuracy"]:.4f}  {r["macro_f1"]:.4f}  '
          f'{xgb_results[h]["accuracy"]:.4f}  {xgb_results[h]["macro_f1"]:.4f}  '
          f'{da:+.4f}  {df1:+.4f}')

a_mlp = np.mean([test_results[h]['accuracy'] for h in HEADS])
f_mlp = np.mean([test_results[h]['macro_f1'] for h in HEADS])
a_xgb = np.mean([xgb_results[h]['accuracy'] for h in HEADS])
f_xgb = np.mean([xgb_results[h]['macro_f1'] for h in HEADS])
print(f'{"Average":<15s}  {a_mlp:.4f}  {f_mlp:.4f}  {a_xgb:.4f}  {f_xgb:.4f}  '
      f'{a_mlp-a_xgb:+.4f}  {f_mlp-f_xgb:+.4f}')

Head             MLP Acc   MLP F1  XGB Acc   XGB F1    Δ Acc     Δ F1
----------------------------------------------------------------------
surface          0.1320  0.1432  0.5512  0.2665  -0.4193  -0.1233
transparency     0.3702  0.3770  0.5846  0.4982  -0.2145  -0.1212
color_family     0.1745  0.1475  0.4533  0.3126  -0.2788  -0.1651
Average          0.2255  0.2226  0.5297  0.3591  -0.3042  -0.1366
